# Mains and one-tricks in NA Master+

**Main question:** Which champions do NA Master+ players main and one-trick (OTP) most, and what are those players' win rates on them?

**Secondary question:** Do mains win more on their champion than other sampled players on the same champion?

**Definitions** (thresholds live in the `settings` CTE of `sql/champion_table.sql` and `sql/player_champions.sql`, and in `src/settings.py`):
- **Sample:** 500 of the 2,000 drawn NA Master+ players, chosen as a repeatable random subset (first 500 by PUUID). This is a prototype-sized sample.
- **Window:** a sampled player's newest 30 ranked solo games, remakes removed. At least 15 games needed to be classified.
- **Main:** 30% or more of the window on one champion. Includes OTPs.
- **OTP:** 70% or more of the window on one champion.
- **Other sampled players:** sampled players who played the champion but are not mains of it. Used for the secondary question (section 4).

Only players whose whole window has been downloaded are included, so results grow as collection runs. Run all cells from the top. In VS Code, pick the `.venv` kernel first.

In [ ]:
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Works whether the notebook runs from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from champion_table import load_champion_table, wilson_interval  # noqa: E402
from db import connect  # noqa: E402
from settings import SAMPLE_SIZE, WINDOW_SIZE  # noqa: E402
from build_site import load_ddragon_champions  # noqa: E402

# participants stores Riot's internal names ("LeeSin"); Data Dragon has the
# display names ("Lee Sin"). build_site.py caches the list in data/ddragon/.
_, ddragon = load_ddragon_champions()
def display_names(champion_ids, fallback_names):
    return [ddragon.get(c, {"name": n})["name"] for c, n in zip(champion_ids, fallback_names)]

def run_sql_file(name):
    with connect() as conn:
        return pd.read_sql_query((ROOT / "sql" / name).read_text(encoding="utf-8"), conn)

def run_sql(query):
    with connect() as conn:
        return pd.read_sql_query(query, conn)

def formatted(df, formats):
    """Copy of df with the given columns turned into formatted strings, for display."""
    out = df.copy()
    for col, f in formats.items():
        out[col] = out[col].map(lambda v: "" if pd.isna(v) else f.format(v))
    return out

# Chart style: light surface, recessive axes and grid, one color per group
COLORS = {"otp": "#eb6834", "main": "#2a78d6", "other": "#1baf7a", "all": "#8a8984"}
LABELS = {"otp": "OTPs", "main": "Mains (incl. OTPs)", "other": "Other sampled players", "all": "All players (context)"}
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "text.color": "#0b0b0b",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e8e7e3", "grid.linewidth": 0.8,
    "axes.titleweight": "bold", "axes.titlesize": 12, "axes.titlelocation": "left",
    "font.size": 10, "figure.dpi": 110,
})

N_BOOTSTRAP = 2000
rng = np.random.default_rng(42)  # fixed seed so results are reproducible

# Charts are also saved as PNGs for the README
IMAGES = ROOT / "images"
IMAGES.mkdir(exist_ok=True)
def save_chart(fig, name):
    fig.savefig(IMAGES / name, dpi=150, bbox_inches="tight")

## 0. Data coverage

In [ ]:
# Match IDs in the sample players' windows (same rule as the SQL files)
sample_window = f'''
    SELECT DISTINCT match_id FROM player_matches
    WHERE recency_rank < {WINDOW_SIZE}
      AND puuid IN (SELECT puuid FROM players ORDER BY puuid LIMIT {SAMPLE_SIZE})
'''
coverage = run_sql(f'''
    SELECT (SELECT COUNT(*) FROM ({sample_window}))                                   AS matches_needed,
           (SELECT COUNT(*) FROM matches WHERE match_id IN ({sample_window}))         AS matches_downloaded,
           (SELECT SUM(is_remake) FROM matches WHERE match_id IN ({sample_window}))   AS remakes,
           (SELECT COUNT(*) FROM unavailable_matches)                                 AS unavailable
''')
display(coverage)

patches = run_sql(f'''
    SELECT patch, COUNT(*) AS matches,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM matches WHERE match_id IN ({sample_window})
    GROUP BY patch ORDER BY matches DESC LIMIT 8
''')
display(patches)

rows = run_sql_file("player_champions.sql")
players = rows.groupby("puuid").agg(tier=("tier", "first"), games=("games", "sum"), wins=("wins", "sum"))
print(f"Classified players (complete window, at least 15 games): {len(players)}")
print(players.tier.value_counts().to_string())

## 1. Who counts as a main or an OTP?

Each player's **top champion share**: games on their most-played champion divided by all games in their window. The dashed lines are the main (30%) and OTP (70%) thresholds.

In [ ]:
top_share = rows.groupby("puuid").share.max()
n = len(top_share)
print(f"Players with a 30%+ champion (have at least one main): {(top_share >= 0.30).sum()} of {n} ({(top_share >= 0.30).mean():.0%})")
print(f"OTPs (70%+ on one champion): {(top_share >= 0.70).sum()} of {n} ({(top_share >= 0.70).mean():.0%})")
print(f"Main rows (player-champion pairs at 30%+): {(rows.label != 'other').sum()}")

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(top_share, bins=np.arange(0, 1.0001, 0.05), color=COLORS["main"], rwidth=0.9)
for x, text in [(0.30, "Main: 30%"), (0.70, "OTP: 70%")]:
    ax.axvline(x, color="#52514e", lw=1.2, ls="--")
    # Label sits just above the plot area so it never collides with a bar
    ax.annotate(text, (x, 1), xycoords=("data", "axes fraction"), xytext=(0, 3),
                textcoords="offset points", ha="center", va="bottom", fontsize=9, color="#52514e")
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlabel("Share of window games on the player's most-played champion")
ax.set_ylabel("Players")
ax.set_title("How concentrated are Master+ champion pools?", pad=18)
ax.grid(axis="x", visible=False)
plt.tight_layout()
save_chart(fig, "2_top_champion_share.png")
plt.show()

## 2. Headline: which champions are mained and one-tricked most?

Each champion's number of mains (30%+ of a player's window) and OTPs (70%+), with those players' win rates on the champion. "All players" is everyone on the champion in the same matches, which shows how the champion itself is doing. Section 3 explains why mains' win rates sit above 50%.

The page (`src/build_site.py`) has the full table for every champion and role; this section shows the top of it.

In [ ]:
champs = load_champion_table()
champs["champion_name"] = display_names(champs.champion_id, champs.champion_name)

TOP_N = 15
top = champs.sort_values(["mains", "otps", "main_games"], ascending=False).head(TOP_N)
cols = ["champion_name", "mains", "otps", "main_games", "main_wr", "otp_games", "otp_wr", "all_games", "all_wr"]
display(formatted(top[cols], {"main_wr": "{:.1%}", "otp_wr": "{:.1%}", "all_wr": "{:.1%}"}))

print(f"Champions with at least one main: {(champs.mains > 0).sum()} of {len(champs)}")
print(f"Champions with at least one OTP: {(champs.otps > 0).sum()}")

fig, ax = plt.subplots(figsize=(7, 0.32 * TOP_N + 1.4))
y = np.arange(TOP_N)[::-1]
mains_only = top.mains - top.otps
# Stacked bars: OTPs first, then the other mains, so the full bar is all mains
ax.barh(y, top.otps, color=COLORS["otp"], label="OTPs (70%+)", height=0.7)
ax.barh(y, mains_only, left=top.otps, color=COLORS["main"], label="Other mains (30% to 69%)", height=0.7,
        edgecolor="#fcfcfb", linewidth=1)
for yi, (n, wr, g) in zip(y, zip(top.mains, top.main_wr, top.main_games)):
    ax.annotate(f"{wr:.0%} WR, {g} games", (n, yi), xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=8.5, color="#52514e")
ax.set_yticks(y, top.champion_name)
ax.set_xlim(0, top.mains.max() * 1.45)  # room for the labels
ax.xaxis.get_major_locator().set_params(integer=True)
ax.set_xlabel("Players (mains include OTPs); label: mains' win rate on the champion")
ax.set_title(f"Most-mained champions, NA Master+ ({len(players)} players)")
ax.grid(axis="y", visible=False)
fig.legend(*ax.get_legend_handles_labels(), loc="lower center", ncol=2, frameon=False, fontsize=9)
plt.tight_layout(rect=(0, 0.06, 1, 1))
save_chart(fig, "1_most_mained_champions.png")
plt.show()

## 3. Why are mains' win rates above 50%?

Our players were picked from the **current** Master+ ladder. Being on the ladder right now means they recently won more than average, so their recent games show a win rate above 50%. Everyone in their matches averages exactly 50% (each game has 5 winners and 5 losers).

So a mains win rate in section 2 describes how these players do, not how strong the champion is; the "all players" win rate is the better guide to the champion itself. Section 4 shows where the extra winning sits: other sampled players win about 50% on champions they do not main, so it comes from games on their mains.

Confidence intervals here and below come from a **player bootstrap**: resample players with replacement, recompute the win rate, repeat 2,000 times, and take the middle 95%. Games from the same player are not independent, so resampling players gives more honest (wider) intervals than treating every game separately.

In [ ]:
def bootstrap_win_rate(games, wins, n_boot=N_BOOTSTRAP):
    """Pooled win rate with a 95% player-bootstrap interval.
    games and wins are arrays with one entry per player."""
    games, wins = np.asarray(games), np.asarray(wins)
    idx = rng.integers(0, len(games), size=(n_boot, len(games)))  # each row is one resample of players
    boot = wins[idx].sum(axis=1) / games[idx].sum(axis=1)
    return wins.sum() / games.sum(), *np.percentile(boot, [2.5, 97.5])

tier_order = ["CHALLENGER", "GRANDMASTER", "MASTER"]
tier_stats = []
for tier in tier_order:
    t = players[players.tier == tier]
    if len(t):
        tier_stats.append((tier.title(), len(t), *bootstrap_win_rate(t.games, t.wins)))
tier_df = pd.DataFrame(tier_stats, columns=["tier", "players", "win_rate", "low", "high"])
display(formatted(tier_df, {"win_rate": "{:.1%}", "low": "{:.1%}", "high": "{:.1%}"}))

fig, ax = plt.subplots(figsize=(7, 2.6))
y = np.arange(len(tier_df))[::-1]
ax.errorbar(tier_df.win_rate, y, xerr=[tier_df.win_rate - tier_df.low, tier_df.high - tier_df.win_rate],
            fmt="o", color=COLORS["main"], ms=8, lw=2, capsize=0)
for yi, (wr, n) in zip(y, zip(tier_df.win_rate, tier_df.players)):
    ax.annotate(f"{wr:.1%}  ({n} players)", (wr, yi), xytext=(0, 9), textcoords="offset points",
                ha="center", fontsize=9, color="#52514e")
ax.axvline(0.5, color=COLORS["all"], lw=1.5, ls="--")
ax.annotate("All players: 50%", (0.5, y.max() + 0.45), ha="left", xytext=(4, 0), textcoords="offset points",
            fontsize=9, color="#52514e")
ax.set_yticks(y, tier_df.tier)
ax.set_ylim(-0.6, y.max() + 0.7)
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlabel("Win rate in their own recent games (95% player-bootstrap interval)")
ax.set_title("Sampled players win more than average")
ax.grid(axis="y", visible=False)
plt.tight_layout()
save_chart(fig, "3_selection_by_tier.png")
plt.show()

## 4. Secondary: do mains win more than other sampled players?

Compares mains with **other sampled players**: players from the same sample who played the champion but are not mains of it. Both groups were picked from the ladder the same way, unlike "all players". Two views:
1. **Pooled win rate per group**, all champions together. Easy to read, but the groups play different champions.
2. **Same-champion difference**: for each champion, mains' win rate minus other sampled players' win rate on that champion, then a weighted average across champions. This answers the actual question (same champion, same kind of player). Each champion's weight is `main_games * other_games / (main_games + other_games)`, the Mantel-Haenszel weight, which gives more say to champions where *both* groups have many games.

Both use the player bootstrap for intervals.

In [ ]:
# One row per player with games and wins in each group (0 if none)
per_player = rows.assign(group=rows.label.replace({"main": "main_only"})).pivot_table(
    index="puuid", columns="group", values=["games", "wins"], aggfunc="sum", fill_value=0)
def player_totals(labels):
    cols = [c for c in labels if ("games", c) in per_player.columns]
    return per_player["games"][cols].sum(axis=1).to_numpy(), per_player["wins"][cols].sum(axis=1).to_numpy()

headline = []
for group, labels in [("otp", ["otp"]), ("main", ["otp", "main_only"]), ("other", ["other"])]:
    g, w = player_totals(labels)
    keep = g > 0  # only players who have games in this group
    headline.append((group, keep.sum(), g.sum(), *bootstrap_win_rate(g[keep], w[keep])))

# All players in the sample's window matches (summed from the champion table in section 2)
all_games, all_wins = champs.all_games.sum(), champs.all_wins.sum()
headline.append(("all", np.nan, all_games, all_wins / all_games, *wilson_interval(all_wins, all_games)))
headline_df = pd.DataFrame(headline, columns=["group", "players", "games", "win_rate", "low", "high"])
headline_df["group"] = headline_df.group.map(LABELS)
display(formatted(headline_df, {"win_rate": "{:.1%}", "low": "{:.1%}", "high": "{:.1%}", "players": "{:.0f}", "games": "{:,.0f}"}))

In [ ]:
# Same-champion difference with a player bootstrap.
player_index = {p: i for i, p in enumerate(rows.puuid.unique())}
champ_index = {c: i for i, c in enumerate(rows.champion_id.unique())}
p_idx = rows.puuid.map(player_index).to_numpy()
c_idx = rows.champion_id.map(champ_index).to_numpy()
games = rows.games.to_numpy()
wins = rows.wins.to_numpy()
n_players, n_champs = len(player_index), len(champ_index)

def same_champion_difference(focus_mask, player_weights):
    """Weighted average over champions of (focus group win rate - other win rate).
    player_weights says how many times each player appears in this resample."""
    w = player_weights[p_idx]
    other_mask = rows.label.to_numpy() == "other"
    def sums(mask):
        return (np.bincount(c_idx, games * w * mask, n_champs), np.bincount(c_idx, wins * w * mask, n_champs))
    fg, fw = sums(focus_mask)
    og, ow = sums(other_mask)
    ok = (fg > 0) & (og > 0)  # champions where both groups have games
    if not ok.any():
        return np.nan  # possible with tiny samples: no champion has both groups
    diff = fw[ok] / fg[ok] - ow[ok] / og[ok]
    weight = fg[ok] * og[ok] / (fg[ok] + og[ok])
    return (weight * diff).sum() / weight.sum()

results = []
for name, mask in [("Mains vs other sampled players", rows.label.isin(["main", "otp"]).to_numpy()),
                   ("OTPs vs other sampled players", (rows.label == "otp").to_numpy())]:
    point = same_champion_difference(mask, np.ones(n_players))
    boot = [same_champion_difference(mask, np.bincount(rng.integers(0, n_players, n_players), minlength=n_players))
            for _ in range(N_BOOTSTRAP)]
    low, high = np.nanpercentile(boot, [2.5, 97.5])  # nan = resample with no usable champion
    results.append((name, point, low, high, np.isnan(boot).sum()))
diff_df = pd.DataFrame(results, columns=["comparison", "difference", "low", "high", "skipped_resamples"])
display(formatted(diff_df, {"difference": "{:+.1%}", "low": "{:+.1%}", "high": "{:+.1%}"}))

In [ ]:
order = ["otp", "main", "other", "all"]
plot_df = pd.DataFrame(headline, columns=["group", "players", "games", "win_rate", "low", "high"]).set_index("group").loc[order]

fig, ax = plt.subplots(figsize=(7, 3))
y = np.arange(len(order))[::-1]
for yi, group in zip(y, order):
    r = plot_df.loc[group]
    ax.errorbar(r.win_rate, yi, xerr=[[r.win_rate - r.low], [r.high - r.win_rate]],
                fmt="o", color=COLORS[group], ms=8, lw=2, capsize=0)
    ax.annotate(f"{r.win_rate:.1%}", (r.win_rate, yi), xytext=(0, 9), textcoords="offset points",
                ha="center", fontsize=9, color="#52514e")
ax.axvline(0.5, color="#c3c2b7", lw=1, zorder=0)
ax.set_yticks(y, [LABELS[g] for g in order])
ax.set_ylim(-0.6, len(order) - 0.3)
ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_xlabel("Win rate (95% interval; player bootstrap, Wilson for all players)")
ax.set_title("Win rate by group, all champions pooled")
ax.grid(axis="y", visible=False)
plt.tight_layout()
save_chart(fig, "4_win_rate_by_group.png")
plt.show()

## 5. Champion by champion: mains vs other sampled players

Mains' win rate next to other sampled players' win rate on the same champion. Only champions with enough data in **both** groups are shown (thresholds below). Intervals are Wilson intervals on games, which are somewhat too narrow because games from the same player are related; treat single champions with caution.

In [ ]:
MIN_MAINS = 3         # at least this many main players on the champion
MIN_MAIN_GAMES = 60   # and this many main games (same as the results page)
MIN_OTHER_GAMES = 60  # and this many games from other sampled players

shown = champs[(champs.mains >= MIN_MAINS) & (champs.main_games >= MIN_MAIN_GAMES)
               & (champs.other_games >= MIN_OTHER_GAMES)].copy()
shown["difference"] = shown.main_wr - shown.other_wr
shown = shown.sort_values("difference")
print(f"Champions meeting the thresholds: {len(shown)} of {len(champs)}")

cols = ["champion_name", "mains", "otps", "main_games", "main_wr", "other_games", "other_wr", "difference", "all_wr"]
display(formatted(shown[cols][::-1], {"main_wr": "{:.1%}", "other_wr": "{:.1%}", "all_wr": "{:.1%}", "difference": "{:+.1%}"}))

if len(shown):
    fig, ax = plt.subplots(figsize=(7, 0.32 * len(shown) + 1.4))
    y = np.arange(len(shown))
    ax.hlines(y, shown.other_wr, shown.main_wr, color="#c3c2b7", lw=2, zorder=1)
    ax.scatter(shown.other_wr, y, s=64, color=COLORS["other"], zorder=2, label=LABELS["other"],
               edgecolor="#fcfcfb", linewidth=2)
    ax.scatter(shown.main_wr, y, s=64, color=COLORS["main"], zorder=3, label=LABELS["main"],
               edgecolor="#fcfcfb", linewidth=2)
    ax.axvline(0.5, color="#c3c2b7", lw=1, zorder=0)
    ax.set_yticks(y, shown.champion_name)
    ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax.set_xlabel("Win rate on the champion")
    ax.set_title("Mains vs other sampled players, same champion")
    # Legend below the chart so it never covers a data point
    fig.legend(*ax.get_legend_handles_labels(), loc="lower center", ncol=2, frameon=False, fontsize=9)
    ax.grid(axis="y", visible=False)
    plt.tight_layout(rect=(0, 0.07, 1, 1))
    save_chart(fig, "5_champions_mains_vs_others.png")
    plt.show()
else:
    print("Not enough data yet. This fills in as collection runs.")

## Notes for the README

- **Selection effect:** sampled players win more than average because they were picked from the current ladder (section 3). Other sampled players win about 50% on non-main champions, so this extra winning sits in mains' games. The comparison cannot separate "mains are better on their champion" from "players climbed to Master+ by winning on their main"; both describe the same games.
- **Players keep champions they win on:** a player who happens to win a lot on a champion may keep playing it and become a "main" partly because of those wins. Some of the main advantage may come from this, not from practice.
- **Multiple patches:** most games are on the latest two patches, but some windows reach back further.
- **Ladder snapshot:** tiers and the player pool are from Oct 31, 2026.
- **Clustering:** headline intervals use a player bootstrap. Per-champion Wilson intervals are somewhat too narrow.